## Load Data

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("notebook-test").getOrCreate()
spark.range(5).show()
spark.stop()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [2]:
# Create SparkSession (entry point for Spark)
spark = (
    SparkSession.builder
    .master("local[*]")              # Use all available CPU cores
    .appName("KMeans_Performance_mini-project")  # Name of the Spark application
    .config("spark.driver.memory", "4g")  # Allocate memory for Spark driver
    .getOrCreate()                   # Create or reuse SparkSession
)

In [3]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    """
    Repo root is defined as the directory that contains `data/dataset.txt`
    """
    start = (start or Path.cwd()).resolve()

    for p in [start, *start.parents]:
        if (p / "data" / "dataset.txt").exists():
            return p

    raise FileNotFoundError(
        "Cannot find repo root. Expected `data/dataset.txt` in a parent directory.\n"
        "Have you run the data preparation script?"
    )

# 1) find repo root robustly
repo_root = find_repo_root()

# 2) read dataset pointer
pointer_file = repo_root / "data" / "dataset.txt"
dataset_dir = Path(pointer_file.read_text(encoding="utf-8").strip())


In [4]:
# 3) load data
csv_path = str(next(dataset_dir.glob("*.csv")))   # Spark ต้องการ string path

sdf = (spark.read
       .option("header", True)
       .option("inferSchema", True)
       .csv(csv_path))

sdf.show(5, truncate=False)
sdf.printSchema()

+---+-------+--------+-------------------+-------------------+-----------------+------------------+-------+-------+------------+-------------------------------------------------------------------------------------+-------------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|ID |Source |Severity|Start_Time         |End_Time           |Start_Lat        |Start_Lng         |End_Lat|End_Lng|Distance(mi)|Description                                                                          |Street                   |City        |County    |State|Zipcode   |Country|Timezone  |Airport_Code|Weather_Timestamp  |T

In [5]:
from pyspark.sql.types import StructField

categorical_cols = [field.name for field in sdf.schema.fields if field.dataType.typeName() == 'string' or field.dataType.typeName() == 'boolean']
numerical_cols = [field.name for field in sdf.schema.fields if field.dataType.typeName() in ['integer','int', 'double', 'bigint']]
datetype_cols = {field.name for field in sdf.schema.fields if field.dataType.typeName() in ['date','Timezone','Weather_Timestamp', 'timestamp']}

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"\nDate/Time columns ({len(datetype_cols)}): {datetype_cols}")

Categorical columns (30): ['ID', 'Source', 'Description', 'Street', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone', 'Airport_Code', 'Wind_Direction', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']

Numerical columns (13): ['Severity', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']

Date/Time columns (3): {'Start_Time', 'Weather_Timestamp', 'End_Time'}


### Numerical (Skewed): 
* Visibility(mi),Humidity(%), Temperature(F) ต้องใช้ "Median" เท่านั้น เพื่อป้องกันไม่ให้ค่าที่เติมถูกดึงโดย Outliers

* Wind Speed: พบค่าพุ่งเกิน 800-1,000 mph ซึ่งเป็นไปไม่ได้ในธรรมชาติ (พายุทอร์นาโดที่รุนแรงที่สุดยังไม่เกิน 300 mph)

In [ ]:
sdf = sdf.filter(cols("Wind_Speed(mph)") <= 150)

In [ ]:
from pyspark.sql.functions import col

# Filter out unrealistic temperature values
sdf = sdf.filter((col("Temperature(F)") >= -40) & (col("Temperature(F)") <= 130))

TypeError: 'list' object is not callable

In [7]:
from pyspark.ml.feature import Imputer

cols = ["Visibility(mi)","Humidity(%)", "Temperature(F)"]
# strategy "median"
imputer = Imputer(inputCols=cols, outputCols=cols).setStrategy("median")

sdf = imputer.fit(sdf).transform(sdf)

* Precipitation(in)	Skewness 85.99	Fill with 0 (สมมติฐานว่าถ้าไม่บันทึกคือไม่มีฝน)

In [9]:
# Handle Precipitation(in) with 0 fill strategy
sdf = sdf.fillna(0, subset=["Precipitation(in)"])

### Time missing <=0.5 -> mode